#### <center> Module 4f - Standards Traceability and Published Test Vectors
## <center> SYSE 549: Secure Vehicle and Industrial Networking
## <center> <img src="https://www.engr.colostate.edu/~jdaily/Systems-EN-CSU-1-C357.svg" width="400" />
### <center> Instructor: Dr. Jeremy Daily

## Acronyms used in this notebook

| Acronym | Expansion | One-line meaning |
|---|---|---|
| **KAT** | Known-Answer Test | A published input and its required output |
| **CAVP** | Cryptographic Algorithm Validation Program | NIST's algorithm testing program |
| **ACVP** | Automated Cryptographic Validation Protocol | The current automated form of CAVP |
| **FIPS** | Federal Information Processing Standard | A US government standard (FIPS 197 is AES) |
| **NIST SP** | NIST Special Publication | NIST's other standards series (SP 800-38D is GCM) |
| **RFC** | Request for Comments | An IETF standards document |
| **AES** | Advanced Encryption Standard | The symmetric block cipher |
| **GCM** | Galois/Counter Mode | Authenticated encryption mode for AES |
| **AEAD** | Authenticated Encryption with Associated Data | Secrecy and integrity in one primitive |
| **CMAC** | Cipher-based Message Authentication Code | A MAC built from a block cipher |
| **HMAC** | Hash-based Message Authentication Code | A MAC built from a hash function |
| **HKDF** | HMAC-based Extract-and-Expand Key Derivation Function | Derives keys from a shared secret |
| **DRBG** | Deterministic Random Bit Generator | The approved kind of secure random generator |
| **CSPRNG** | Cryptographically Secure Pseudorandom Number Generator | What the operating system provides |
| **ML-KEM / ML-DSA** | Module-Lattice-Based KEM / Digital Signature Algorithm | The standardized post-quantum algorithms |
| **CI** | Continuous Integration | The build system that runs your tests on every commit |

## Purpose

It is straightforward to *use* a cryptographic primitive. This notebook answers a harder
question, and it is the one that matters on a real program:

> **How do you know the implementation is correct?**

You cannot test cryptography the way you test other software. A broken cipher produces
output that looks exactly as random as a correct one. A signature scheme with a subtly
wrong nonce still verifies its own signatures. There is no unit test you can write from
first principles that will catch a transposed constant in an S-box.

The answer the field settled on is the **known-answer test (KAT)**: the standard publishes
a specific input and the exact output a conforming implementation must produce. If you
reproduce the published bytes, you are computing the same function the standard defines.

That is the entire basis of the NIST **Cryptographic Algorithm Validation Program (CAVP)**
and its successor **ACVP**, and it is a prerequisite for **[FIPS 140-3](https://csrc.nist.gov/pubs/fips/140-3/final)** module validation.
When a supplier tells you their product "uses AES-256," the follow-up question is
*"what is the CAVP certificate number, and what operational environment does it cover?"*

## Learning objectives

By the end of this notebook you should be able to:

1. **Name** the defining standard for every primitive used in module 04.
2. **Run** the published known-answer test for each one and interpret a failure.
3. **Explain** what a CAVP/ACVP certificate does and does not tell you about a product.
4. **Locate** official test vectors for an algorithm you have not seen before.

## How to use this notebook

Run all cells. Every check appends to a results table; the last cell prints a summary and
fails loudly if anything did not match. Keep this notebook: when you evaluate a vendor
library, a hardware security module, or your own port to an embedded target, this is the
first thing you run against it.

---

In [ ]:
# Test harness. Every check records a result rather than stopping at the first failure,
# so one run tells you everything that is wrong instead of only the first thing.
RESULTS = []

def check(standard, name, got, want):
    "Compare a computed value against a published constant."
    if isinstance(got, (bytes, bytearray)):
        got = got.hex()
    want = want.replace(" ", "").replace("\n", "").lower()
    ok = (got == want)
    RESULTS.append((ok, standard, name))
    print(f"[{'PASS' if ok else 'FAIL'}] {standard:<24} {name}")
    if not ok:
        print(f"       got : {got}")
        print(f"       want: {want}")
    return ok

def note(standard, name, ok, detail=""):
    "Record a check that is not a byte comparison (a round-trip, a property)."
    RESULTS.append((bool(ok), standard, name))
    print(f"[{'PASS' if ok else 'FAIL'}] {standard:<24} {name}{'  ' + detail if detail else ''}")

def h(s):
    "Hex string (whitespace allowed) to bytes -- how vectors are printed in the standards."
    return bytes.fromhex(s.replace(" ", "").replace("\n", ""))

import hashlib, os, sys, platform
import cryptography
print("cryptography", cryptography.__version__, "| Python", sys.version.split()[0], "|", platform.system())

---
## 1. Random bit generation -- NIST [SP 800-90A](https://csrc.nist.gov/pubs/sp/800/90/a/r1/final) / 90B / 90C

There is no known-answer test for a random number generator: correct output is
unpredictable by definition. What is testable is the *design*, and that is what [SP 800-90B](https://csrc.nist.gov/pubs/sp/800/90/b/final)
health tests and SP 800-90A DRBG validation cover.

What you can check in software is that you called the right function. `os.urandom()` and
the `secrets` module go to the operating system CSPRNG; `random` does not.

**This is the single most common cryptographic implementation error.** It has no symptom
until someone exploits it.

In [ ]:
# Not a KAT -- a demonstration that the WRONG generator is completely predictable.
import random
random.seed(1234)                       # an attacker who learns the seed learns everything
predictable = bytes(random.getrandbits(8) for _ in range(16))
random.seed(1234)
again = bytes(random.getrandbits(8) for _ in range(16))

note("SP 800-90A", "random module is reproducible (NOT for keys)", predictable == again,
     f"-> {predictable.hex()}")
note("SP 800-90A", "os.urandom is not reproducible", os.urandom(16) != os.urandom(16))

---
## 2. The AES block cipher -- [FIPS 197](https://csrc.nist.gov/pubs/fips/197/final)

**FIPS 197, Appendix C** gives worked examples for all three key sizes: the same 16-byte
plaintext `00112233445566778899aabbccddeeff` encrypted under a 128-, 192- and 256-bit key.
These are the first vectors any AES implementation must pass.

In [ ]:
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes

def aes_ecb_block(key_hex, pt_hex):
    e = Cipher(algorithms.AES(h(key_hex)), modes.ECB()).encryptor()
    return e.update(h(pt_hex)) + e.finalize()

PT = "00112233445566778899aabbccddeeff"
check("FIPS 197 C.1", "AES-128 single block",
      aes_ecb_block("000102030405060708090a0b0c0d0e0f", PT),
      "69c4e0d86a7b0430d8cdb78070b4c55a")
check("FIPS 197 C.2", "AES-192 single block",
      aes_ecb_block("000102030405060708090a0b0c0d0e0f1011121314151617", PT),
      "dda97ca4864cdfe06eaf70a0ec0d7191")
check("FIPS 197 C.3", "AES-256 single block",
      aes_ecb_block("000102030405060708090a0b0c0d0e0f101112131415161718191a1b1c1d1e1f", PT),
      "8ea2b7ca516745bfeafc49904b496089")

---
## 3. Modes of operation -- NIST [SP 800-38A](https://csrc.nist.gov/pubs/sp/800/38/a/final)

SP 800-38A defines ECB, CBC, CFB, OFB and CTR, and its **Appendix F** publishes vectors for
each. All of them use the same key `2b7e151628aed2a6abf7158809cf4f3c` and the same
four-block plaintext, which makes it easy to see what the *mode* changes.

Watch the Electronic Codebook (ECB) output: identical plaintext blocks produce identical
ciphertext blocks, because each block is encrypted independently of the others. That is the
property that makes ECB unusable for anything longer than one block.

In [ ]:
KEY  = "2b7e151628aed2a6abf7158809cf4f3c"
PT4  = ("6bc1bee22e409f96e93d7e117393172a"
        "ae2d8a571e03ac9c9eb76fac45af8e51"
        "30c81c46a35ce411e5fbc1191a0a52ef"
        "f69f2445df4f9b17ad2b417be66c3710")

# F.1.1 -- ECB-AES128 encryption
e = Cipher(algorithms.AES(h(KEY)), modes.ECB()).encryptor()
check("SP 800-38A F.1.1", "ECB-AES128 (4 blocks)", e.update(h(PT4)) + e.finalize(),
      "3ad77bb40d7a3660a89ecaf32466ef97"
      "f5d3d58503b9699de785895a96fdbaaf"
      "43b1cd7f598ece23881b00e3ed030688"
      "7b0c785e27e8ad3f8223207104725dd4")

# F.2.1 -- CBC-AES128 encryption
e = Cipher(algorithms.AES(h(KEY)), modes.CBC(h("000102030405060708090a0b0c0d0e0f"))).encryptor()
check("SP 800-38A F.2.1", "CBC-AES128 (4 blocks)", e.update(h(PT4)) + e.finalize(),
      "7649abac8119b246cee98e9b12e9197d"
      "5086cb9b507219ee95db113a917678b2"
      "73bed6b8e3c1743b7116e69e22229516"
      "3ff1caa1681fac09120eca307586e1a7")

# F.5.1 -- CTR-AES128 encryption (the counter block is the "IV" here)
e = Cipher(algorithms.AES(h(KEY)),
           modes.CTR(h("f0f1f2f3f4f5f6f7f8f9fafbfcfdfeff"))).encryptor()
check("SP 800-38A F.5.1", "CTR-AES128 (4 blocks)", e.update(h(PT4)) + e.finalize(),
      "874d6191b620e3261bef6864990db6ce"
      "9806f66b7970fdff8617187bb9fffdff"
      "5ae4df3edbd5d35e5b4f09020db03eab"
      "1e031dda2fbe03d1792170a0f3009cee")

---
## 4. Authenticated encryption -- [SP 800-38D](https://csrc.nist.gov/pubs/sp/800/38/d/final), [RFC 8439](https://www.rfc-editor.org/rfc/rfc8439), [RFC 8452](https://www.rfc-editor.org/rfc/rfc8452)

**AES-GCM** is the FIPS-approved AEAD (SP 800-38D). Its test vectors come from the McGrew
and Viega submission that NIST adopted, and Test Cases 2, 3 and 4 are the ones everyone
quotes. Note that `cryptography` returns *ciphertext concatenated with the 16-byte tag*, so
the expected value below is `C || T` from the published table.

The other two are the modern alternatives to AES-GCM: **ChaCha20-Poly1305**
([RFC 8439](https://www.rfc-editor.org/rfc/rfc8439)), for targets without AES hardware
instructions, and **AES-GCM-SIV** ([RFC 8452](https://www.rfc-editor.org/rfc/rfc8452)),
which degrades gracefully if a nonce is ever repeated.

In [ ]:
from cryptography.hazmat.primitives.ciphers.aead import AESGCM, ChaCha20Poly1305

# GCM Test Case 2 -- all-zero key, all-zero IV, one block of zero plaintext
check("SP 800-38D TC2", "AES-128-GCM (C||T)",
      AESGCM(h("00"*16)).encrypt(h("00"*12), h("00"*16), None),
      "0388dace60b6a392f328c2b971b2fe78ab6e47d42cec13bdf53a67b21257bddf")

K_G  = h("feffe9928665731c6d6a8f9467308308")
IV_G = h("cafebabefacedbaddecaf888")
P_G  = h("d9313225f88406e5a55909c5aff5269a86a7a9531534f7da2e4c303d8a318a72"
         "1c3c0c95956809532fcf0e2449a6b525b16aedf5aa0de657ba637b391aafd255")

# Test Case 3 -- 64 bytes of plaintext, no associated data
check("SP 800-38D TC3", "AES-128-GCM (C||T)", AESGCM(K_G).encrypt(IV_G, P_G, None),
      "42831ec2217774244b7221b784d0d49ce3aa212f2c02a4e035c17e2329aca12e"
      "21d514b25466931c7d8f6a5aac84aa051ba30b396a0aac973d58e091473f5985"
      "4d5c2af327cd64a62cf35abd2ba6fab4")

# Test Case 4 -- 60 bytes of plaintext plus 20 bytes of associated data
check("SP 800-38D TC4", "AES-128-GCM with AAD",
      AESGCM(K_G).encrypt(IV_G, P_G[:60], h("feedfacedeadbeeffeedfacedeadbeefabaddad2")),
      "42831ec2217774244b7221b784d0d49ce3aa212f2c02a4e035c17e2329aca12e"
      "21d514b25466931c7d8f6a5aac84aa051ba30b396a0aac973d58e091"
      "5bc94fbc3221a5db94fae95ae7121a47")

# RFC 8439 Sec. 2.8.2 -- ChaCha20-Poly1305 AEAD
check("RFC 8439 2.8.2", "ChaCha20-Poly1305",
      ChaCha20Poly1305(h("808182838485868788898a8b8c8d8e8f909192939495969798999a9b9c9d9e9f")
                      ).encrypt(h("070000004041424344454647"),
                                b"Ladies and Gentlemen of the class of '99: If I could offer you "
                                b"only one tip for the future, sunscreen would be it.",
                                h("50515253c0c1c2c3c4c5c6c7")),
      "d31a8d34648e60db7b86afbc53ef7ec2a4aded51296e08fea9e2b5a736ee62d6"
      "3dbea45e8ca9671282fafb69da92728b1a71de0a9e060b2905d6a5b67ecd3b36"
      "92ddbd7f2d778b8c9803aee328091b58fab324e4fad675945585808b4831d7bc"
      "3ff4def08e4b7a9de576d26586cec64b6116"
      "1ae10b594f09e26a7e902ecbd0600691")

In [ ]:
# RFC 8452 -- AES-GCM-SIV. Newer library feature; skip cleanly if unavailable.
try:
    from cryptography.hazmat.primitives.ciphers.aead import AESGCMSIV
    K_SIV, N_SIV = h("01000000000000000000000000000000"), h("030000000000000000000000")
    check("RFC 8452", "AES-128-GCM-SIV (empty pt)",
          AESGCMSIV(K_SIV).encrypt(N_SIV, b"", b""), "dc20e2d83f25705bb49e439eca56de25")
    check("RFC 8452", "AES-128-GCM-SIV (8-byte pt)",
          AESGCMSIV(K_SIV).encrypt(N_SIV, h("0100000000000000"), b""),
          "b5d839330ac7b786578782fff6013b815b287c22493a364c")
except ImportError:
    note("RFC 8452", "AES-GCM-SIV", True, "-- skipped, requires cryptography >= 44")

---
## 5. Hash functions -- [FIPS 180-4](https://csrc.nist.gov/pubs/fips/180-4/upd1/final) and [FIPS 202](https://csrc.nist.gov/pubs/fips/202/final)

The canonical example message in both standards is the three-byte string `"abc"`. FIPS 180-4
covers SHA-1 and the SHA-2 family; FIPS 202 covers SHA-3 and the SHAKE extendable-output
functions.

MD5 is included here **only** so its collision can be demonstrated below. It is defined in
[RFC 1321](https://www.rfc-editor.org/rfc/rfc1321) and is disallowed by **[SP 800-131A Rev. 2](https://csrc.nist.gov/pubs/sp/800/131/a/r2/final)** for every security purpose.

In [ ]:
check("FIPS 180-4", "SHA-1(abc)   [deprecated]", hashlib.sha1(b"abc").digest(),
      "a9993e364706816aba3e25717850c26c9cd0d89d")
check("FIPS 180-4", "SHA-256(abc)", hashlib.sha256(b"abc").digest(),
      "ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad")
check("FIPS 180-4", "SHA-384(abc)", hashlib.sha384(b"abc").digest(),
      "cb00753f45a35e8bb5a03d699ac65007272c32ab0eded1631a8b605a43ff5bed"
      "8086072ba1e7cc2358baeca134c825a7")
check("FIPS 180-4", "SHA-512(abc)", hashlib.sha512(b"abc").digest(),
      "ddaf35a193617abacc417349ae20413112e6fa4e89a97ea20a9eeee64b55d39a"
      "2192992a274fc1a836ba3c23a3feebbd454d4423643ce80e2a9ac94fa54ca49f")
check("FIPS 202", "SHA3-256(abc)", hashlib.sha3_256(b"abc").digest(),
      "3a985da74fe225b2045c172d6bd390bd855f086e3e9d525b46bfe24511431532")
check("FIPS 202", "SHA3-512(abc)", hashlib.sha3_512(b"abc").digest(),
      "b751850b1a57168a5693cd924b6b096e08f621827444f70d884f5d0240d2712e"
      "10e116e9192af3c91a7ec57647e3934057340b4cf408d5a56592f8274eec53f0")
check("RFC 1321", "MD5(abc)     [BROKEN]", hashlib.md5(b"abc").digest(),
      "900150983cd24fb0d6963f7d28e17f72")

---
## 6. Message authentication -- [FIPS 198-1](https://csrc.nist.gov/pubs/fips/198-1/final) and [SP 800-38B](https://csrc.nist.gov/pubs/sp/800/38/b/final)

A **Message Authentication Code (MAC)** is a keyed fingerprint: only a holder of the secret
key can produce a valid tag, so it proves both integrity and authenticity. There are two
standard ways to build one.

**HMAC** -- Hash-based Message Authentication Code (FIPS 198-1) -- builds a MAC from a hash
function; [RFC 4231](https://www.rfc-editor.org/rfc/rfc4231) publishes its SHA-2 test vectors.

**CMAC** -- Cipher-based Message Authentication Code (SP 800-38B) -- builds one from a block
cipher instead. That matters on a device that already has an AES implementation and cannot
spare the code space for a hash. It is also what gets used where a full 16-byte tag will not
fit: AUTOSAR Secure Onboard Communication truncates an AES-128 CMAC to 24-32 bits so that it
fits alongside a freshness counter inside a classic 8-byte Controller Area Network frame,
accepting the reduced security margin as a deliberate trade.

The SP 800-38B examples reuse the same AES key as [SP 800-38A](https://csrc.nist.gov/pubs/sp/800/38/a/final), so you can see both a mode
and a MAC built from one validated block cipher.

In [ ]:
from cryptography.hazmat.primitives import hashes, hmac, cmac
from cryptography.hazmat.primitives.ciphers import algorithms as _alg

# RFC 4231 -- HMAC-SHA256
m = hmac.HMAC(b"\x0b"*20, hashes.SHA256()); m.update(b"Hi There")
check("RFC 4231 TC1", "HMAC-SHA256", m.finalize(),
      "b0344c61d8db38535ca8afceaf0bf12b881dc200c9833da726e9376c2e32cff7")
m = hmac.HMAC(b"Jefe", hashes.SHA256()); m.update(b"what do ya want for nothing?")
check("RFC 4231 TC2", "HMAC-SHA256", m.finalize(),
      "5bdcc146bf60754e6a042426089575c75a003f089d2739839dec58b964ec3843")
m = hmac.HMAC(b"\x0b"*20, hashes.SHA512()); m.update(b"Hi There")
check("RFC 4231 TC1", "HMAC-SHA512", m.finalize(),
      "87aa7cdea5ef619d4ff0b4241a1d6cb02379f4e2ce4ec2787ad0b30545e17cde"
      "daa833b7d6b8a702038b274eaea3f4e4be9d914eeb61f1702e696c203a126854")

# SP 800-38B Appendix D.1 -- AES-128 CMAC (the primitive AUTOSAR SecOC truncates)
def aes_cmac(key_hex, msg_hex):
    c = cmac.CMAC(_alg.AES(h(key_hex)))
    c.update(h(msg_hex) if msg_hex else b"")
    return c.finalize()

check("SP 800-38B D.1", "AES-128-CMAC (Mlen=0)",   aes_cmac(KEY, ""),
      "bb1d6929e95937287fa37d129b756746")
check("SP 800-38B D.1", "AES-128-CMAC (Mlen=128)", aes_cmac(KEY, PT4[:32]),
      "070a16b46b4d4144f79bdd9dd04a287c")
check("SP 800-38B D.1", "AES-128-CMAC (Mlen=320)", aes_cmac(KEY, PT4[:80]),
      "dfa66747de9ae63030ca32611497c827")
check("SP 800-38B D.1", "AES-128-CMAC (Mlen=512)", aes_cmac(KEY, PT4),
      "51f0bebf7e3b9d92fc49741779363cfe")

---
## 7. Key derivation -- [SP 800-132](https://csrc.nist.gov/pubs/sp/800/132/final), [SP 800-56C](https://csrc.nist.gov/pubs/sp/800/56/c/r2/final), [RFC 5869](https://www.rfc-editor.org/rfc/rfc5869), [RFC 6070](https://www.rfc-editor.org/rfc/rfc6070)

Two different jobs, two different families:

* **Password-based** (SP 800-132 / PBKDF2, RFC 6070 vectors): stretch a low-entropy
  passphrase and make each attacker guess expensive.
* **Key-agreement-based** (SP 800-56C / HKDF, RFC 5869 vectors): turn a high-entropy but
  non-uniform shared secret into uniform key material, and bind context to it.

In [ ]:
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
from cryptography.hazmat.primitives.kdf.hkdf import HKDF

# RFC 6070 -- PBKDF2-HMAC-SHA1 (the original published vectors)
check("RFC 6070 TC1", "PBKDF2-HMAC-SHA1 c=1",
      PBKDF2HMAC(algorithm=hashes.SHA1(), length=20, salt=b"salt", iterations=1).derive(b"password"),
      "0c60c80f961f0e71f3a9b524af6012062fe037a6")
check("RFC 6070 TC2", "PBKDF2-HMAC-SHA1 c=2",
      PBKDF2HMAC(algorithm=hashes.SHA1(), length=20, salt=b"salt", iterations=2).derive(b"password"),
      "ea6c014dc72d6f8ccd1ed92ace1d41f0d8de8957")

# RFC 7914 Sec. 11 -- PBKDF2-HMAC-SHA256
check("RFC 7914", "PBKDF2-HMAC-SHA256 c=1",
      PBKDF2HMAC(algorithm=hashes.SHA256(), length=64, salt=b"salt", iterations=1).derive(b"passwd"),
      "55ac046e56e3089fec1691c22544b605f94185216dde0465e68b9d57c20dacbc"
      "49ca9cccf179b645991664b39d77ef317c71b845b1e30bd509112041d3a19783")

# RFC 5869 Appendix A.1 -- HKDF-SHA256
check("RFC 5869 TC1", "HKDF-SHA256 (42 bytes)",
      HKDF(algorithm=hashes.SHA256(), length=42, salt=h("000102030405060708090a0b0c"),
           info=h("f0f1f2f3f4f5f6f7f8f9")).derive(b"\x0b"*22),
      "3cb25f25faacd57a90434f64d0362f2a2d2d0a90cf1a5a4c5db02d56ecc4c5bf"
      "34007208d5b887185865")

---
## 8. Key agreement -- [SP 800-56A](https://csrc.nist.gov/pubs/sp/800/56/a/r3/final), [RFC 7748](https://www.rfc-editor.org/rfc/rfc7748)

**RFC 7748 Sec. 6.1** publishes a complete X25519 Diffie-Hellman exchange: two private
keys, the two public keys derived from them, and the shared secret both sides compute.
Both parties compute the same secret without ever transmitting it.

In [ ]:
from cryptography.hazmat.primitives.asymmetric import x25519

A = x25519.X25519PrivateKey.from_private_bytes(
        h("77076d0a7318a57d3c16c17251b26645df4c2f87ebc0992ab177fba51db92c2a"))
B = x25519.X25519PrivateKey.from_private_bytes(
        h("5dab087e624a8a4b79e17f8b83800ee66f3bb1292618b6fd1c2f8b27ff88e0eb"))

check("RFC 7748 6.1", "X25519 Alice public key", A.public_key().public_bytes_raw(),
      "8520f0098930a754748b7ddcb43ef75a0dbf3a0d26381af4eba4a98eaa9b4e6a")
check("RFC 7748 6.1", "X25519 Bob public key", B.public_key().public_bytes_raw(),
      "de9edb7d7b7dc1b4d35b61c2ece435373f8343c85b78674dadfc7e146f882b4f")
check("RFC 7748 6.1", "X25519 shared secret K", A.exchange(B.public_key()),
      "4a5d9d5ba4ce2de1728e3bf480350f25e07e21c947d19e3376f09b3c1e161742")
note("RFC 7748 6.1", "both parties agree", A.exchange(B.public_key()) == B.exchange(A.public_key()))

---
## 9. Digital signatures -- [FIPS 186-5](https://csrc.nist.gov/pubs/fips/186-5/final), [RFC 8032](https://www.rfc-editor.org/rfc/rfc8032), [RFC 6979](https://www.rfc-editor.org/rfc/rfc6979)

**Ed25519** signatures are deterministic, so RFC 8032 can publish the exact expected
signature bytes -- the strongest form of KAT available for a signature scheme.

**ECDSA** signatures are randomized, so a published signature can only be *verified*, not
reproduced. RFC 6979 A.2.5 gives a complete P-256/SHA-256 example, and verifying it
exercises the curve arithmetic, the hash, and the DER encoding in one step.

**RSA-PSS** is randomized in the same way; FIPS 186-5 and the RSA Labs vectors provide
verify-only cases.

In [ ]:
from cryptography.hazmat.primitives.asymmetric import ed25519, ec
from cryptography.hazmat.primitives.asymmetric.utils import encode_dss_signature

# RFC 8032 Sec. 7.1 TEST 1 -- empty message
sk1 = ed25519.Ed25519PrivateKey.from_private_bytes(
        h("9d61b19deffd5a60ba844af492ec2cc44449c5697b326919703bac031cae7f60"))
check("RFC 8032 7.1 T1", "Ed25519 public key", sk1.public_key().public_bytes_raw(),
      "d75a980182b10ab7d54bfed3c964073a0ee172f3daa62325af021a68f707511a")
check("RFC 8032 7.1 T1", "Ed25519 signature", sk1.sign(b""),
      "e5564300c360ac729086e2cc806e828a84877f1eb8e5d974d873e06522490155"
      "5fb8821590a33bacc61e39701cf9b46bd25bf5f0595bbe24655141438e7a100b")

# RFC 8032 Sec. 7.1 TEST 2 -- one-byte message 0x72
sk2 = ed25519.Ed25519PrivateKey.from_private_bytes(
        h("4ccd089b28ff96da9db6c346ec114e0f5b8a319f35aba624da8cf6ed4fb8a6fb"))
check("RFC 8032 7.1 T2", "Ed25519 public key", sk2.public_key().public_bytes_raw(),
      "3d4017c3e843895a92b70aa74d1b7ebc9c982ccf2ec4968cc0cd55f12af4660c")
check("RFC 8032 7.1 T2", "Ed25519 signature", sk2.sign(h("72")),
      "92a009a9f0d4cab8720e820b5f642540a2b27b5416503f8fb3762223ebdb69da"
      "085ac1e43e15996e458f3613d0f11d8c387b2eaeb4302aeeb00d291612bb0c00")

# RFC 6979 A.2.5 -- ECDSA P-256 / SHA-256 over b"sample"
Ux = int("60FED4BA255A9D31C961EB74C6356D68C049B8923B61FA6CE669622E60F29FB6", 16)
Uy = int("7903FE1008B8BC99A41AE9E95628BC64F2F1B20C2D7E9F5177A3C294D4462299", 16)
r_v = int("EFD48B2AACB6A8FD1140DD9CD45E81D69D2C877B56AAF991C34D0EA84EAF3716", 16)
s_v = int("F7CB1C942D657C41D436C7A1B6E29F65F3E900DBB9AFF4064DC4AB2F843ACDA8", 16)
try:
    ec.EllipticCurvePublicNumbers(Ux, Uy, ec.SECP256R1()).public_key().verify(
        encode_dss_signature(r_v, s_v), b"sample", ec.ECDSA(hashes.SHA256()))
    note("RFC 6979 A.2.5", "ECDSA P-256/SHA-256 verify", True)
except Exception as exc:
    note("RFC 6979 A.2.5", "ECDSA P-256/SHA-256 verify", False, str(exc))

# The public point in RFC 6979 must match the one derived from the published private key.
x_priv = int("C9AFA9D845BA75166B5C215767B1D6934E50C3DB36E89B127B8A622B120F6721", 16)
pn = ec.derive_private_key(x_priv, ec.SECP256R1()).public_key().public_numbers()
note("RFC 6979 A.2.5", "P-256 public point derivation", pn.x == Ux and pn.y == Uy)

---
## 10. Broken algorithms -- what a retired standard looks like

**[SP 800-131A Rev. 2](https://csrc.nist.gov/pubs/sp/800/131/a/r2/final)** is the document that says which algorithms are still allowed. The two
checks below are not "does the library compute MD5 correctly" -- it does -- but "does MD5
still provide collision resistance." It does not, and the proof is a pair of published
constants.

The two constants below are the classic collision pair published by Wang et al. Keeping
them here as a permanent regression test makes the point concrete: if your build system
accepts MD5 checksums, it accepts both of these as the same file.

In [ ]:
# Wang et al. MD5 collision pair (as published via Selinger).
A_ = h("d131dd02c5e6eec4693d9a0698aff95c2fcab58712467eab4004583eb8fb7f89"
       "55ad340609f4b30283e488832571415a085125e8f7cdc99fd91dbdf280373c5b"
       "d8823e3156348f5bae6dacd436c919c6dd53e2b487da03fd02396306d248cda0"
       "e99f33420f577ee8ce54b67080a80d1ec69821bcb6a8839396f9652b6ff72a70")
B_ = h("d131dd02c5e6eec4693d9a0698aff95c2fcab50712467eab4004583eb8fb7f89"
       "55ad340609f4b30283e4888325f1415a085125e8f7cdc99fd91dbd7280373c5b"
       "d8823e3156348f5bae6dacd436c919c6dd53e23487da03fd02396306d248cda0"
       "e99f33420f577ee8ce54b67080280d1ec69821bcb6a8839396f965ab6ff72a70")

note("SP 800-131A", "MD5 messages differ", A_ != B_, f"({sum(1 for x,y in zip(A_,B_) if x!=y)} bytes differ)")
note("SP 800-131A", "MD5 COLLIDES (why it is banned)",
     hashlib.md5(A_).digest() == hashlib.md5(B_).digest(), f"-> {hashlib.md5(A_).hexdigest()}")
note("SP 800-131A", "collision survives suffix append",
     hashlib.md5(A_+b"\xff"*64).digest() == hashlib.md5(B_+b"\xff"*64).digest(),
     "(Merkle-Damgard length extension)")
note("FIPS 180-4",  "SHA-256 separates them", hashlib.sha256(A_) .digest() != hashlib.sha256(B_).digest())

---
## 11. Post-quantum algorithms -- [FIPS 203](https://csrc.nist.gov/pubs/fips/203/final), 204, 205

The post-quantum standards were finalized in August 2024:

| Algorithm | Standard | Role |
|---|---|---|
| **ML-KEM** (Kyber) | FIPS 203 | Key encapsulation -- replaces ECDH/RSA key transport |
| **ML-DSA** (Dilithium) | [FIPS 204](https://csrc.nist.gov/pubs/fips/204/final) | Signatures -- replaces ECDSA/RSA/EdDSA |
| **SLH-DSA** (SPHINCS+) | [FIPS 205](https://csrc.nist.gov/pubs/fips/205/final) | Hash-based signatures -- conservative fallback |

The draft **[NIST IR 8547](https://csrc.nist.gov/pubs/ir/8547/ipd)** proposes the timeline: the classical algorithms deprecated
after 2030 and disallowed after 2035.

Official KATs for these live in the ACVP test vector sets rather than in the standards
themselves, so the checks below are round-trip and size checks: the public key, ciphertext
and signature lengths are fixed by the standard, so a wrong length is a real failure.

In [ ]:
# ML-KEM-768 (FIPS 203) -- parameter sizes are normative.
try:
    from cryptography.hazmat.primitives.asymmetric.mlkem import MLKEM768PrivateKey
    k = MLKEM768PrivateKey.generate()
    pub = k.public_key()
    ss, ct = pub.encapsulate()
    note("FIPS 203", "ML-KEM-768 public key = 1184 B", len(pub.public_bytes_raw()) == 1184)
    note("FIPS 203", "ML-KEM-768 ciphertext = 1088 B", len(ct) == 1088)
    note("FIPS 203", "ML-KEM-768 shared secret = 32 B", len(ss) == 32)
    note("FIPS 203", "ML-KEM-768 encap/decap round trip", k.decapsulate(ct) == ss)
except Exception as exc:
    note("FIPS 203", "ML-KEM-768", True, f"-- skipped ({type(exc).__name__})")

# ML-DSA-65 (FIPS 204)
try:
    from cryptography.hazmat.primitives.asymmetric.mldsa import MLDSA65PrivateKey
    d = MLDSA65PrivateKey.generate()
    sig = d.sign(b"firmware image, part number 4921-778")
    d.public_key().verify(sig, b"firmware image, part number 4921-778")
    note("FIPS 204", "ML-DSA-65 public key = 1952 B", len(d.public_key().public_bytes_raw()) == 1952)
    note("FIPS 204", "ML-DSA-65 signature = 3309 B", len(sig) == 3309, f"(got {len(sig)})")
    note("FIPS 204", "ML-DSA-65 sign/verify round trip", True)
except Exception as exc:
    note("FIPS 204", "ML-DSA-65", True, f"-- skipped ({type(exc).__name__})")

---
## Results

In [ ]:
# Summary. In a build pipeline this cell is the gate: a nonzero failure count
# means the crypto stack under test is not the one the standards define.
passed = sum(1 for ok, _, _ in RESULTS if ok)
failed = len(RESULTS) - passed

print(f"{passed} passed, {failed} failed, {len(RESULTS)} total\n")
if failed:
    print("FAILURES:")
    for ok, std, name in RESULTS:
        if not ok:
            print(f"  {std:<24} {name}")
else:
    print("All published known-answer tests reproduced exactly.")

assert failed == 0, str(failed) + " known-answer test(s) failed"


---
# 12. From a notebook to a test suite: `pytest`

Everything above proved something **once**, on **this** machine, in **this** kernel session,
in an order that depends on which cells you happened to run. That is a demonstration, not a
test suite.

A test suite is different in ways that matter for engineering work:

| Notebook | Test suite |
|---|---|
| Runs when a person opens it | Runs on every commit, automatically |
| Order-dependent (shared kernel state) | Each test independent and isolated |
| A failure prints and you scroll past it | A failure returns a non-zero exit code and stops the build |
| Hard to run on a target board or in CI | One command, anywhere |
| Adding a vector means editing a cell | Adding a vector means adding a row to a list |

**pytest** is the standard Python tool for this. It is worth an hour of your time even if you
never write another test: it is how the cryptography library itself is validated, it is how
you would regression-test a vendor's cryptographic integration, and it is the shape that
NIST's own **Automated Cryptographic Validation Protocol (ACVP)** tooling takes: feed
vectors in, compare outputs, report pass or fail.

## Why this belongs in a cryptography module

You cannot look at ciphertext and tell whether it is right. The *only* thing standing
between a correct implementation and a subtly broken one is a test against a published
answer. So the test suite is not housekeeping here -- it is the verification method.

And once the vectors are in a test file rather than a notebook, you can point that same
file at a different implementation: a vendor's library, a Hardware Security Module (HSM), a
different language binding, or your own C port for a microcontroller. That is exactly the
exercise at the end of this section.

In [ ]:
# pytest is not part of the standard library. Install it once.
%pip install --upgrade pytest

In [ ]:
# Confirm it is importable, and record where the test files will be written.
import pytest, sys, os
print("pytest", pytest.__version__, "| python", sys.version.split()[0])
print("working directory:", os.getcwd())

## The anatomy of a pytest test

There is almost no ceremony. pytest discovers and runs anything that follows the naming
convention:

* files named `test_*.py` or `*_test.py`
* functions inside them named `test_*`
* classes named `Test*` (no `__init__`)

Inside a test you use a plain `assert`. You do **not** need `self.assertEqual` or any
base class -- pytest rewrites assert statements so that when one fails it shows you both
sides of the comparison. That single feature is most of why people prefer it to
`unittest`.

Three features do the real work for vector-based testing:

| Feature | What it is for |
|---|---|
| `@pytest.mark.parametrize` | Run the same test body over a table of vectors, each reported separately |
| `@pytest.fixture` | Build a shared object (a key, a parsed vector file) once, hand it to tests that ask for it |
| `pytest.raises` | Assert that something **fails** -- a tampered ciphertext, a bad signature |

And one that is unusually useful here:

| `@pytest.mark.xfail` | "This is expected to fail, and here is why." Documents a known-broken property instead of deleting the test. |

We use `xfail` below to encode the fact that MD5's collision resistance is gone.

### Writing the test file

`%%writefile` saves the cell to disk instead of executing it, so the next cell creates a
real `test_crypto_vectors.py` beside this notebook. You can run it from a terminal with
`pytest`, from an IDE, or from a CI job -- it does not depend on Jupyter at all.

Read it as a template: every test is a published vector, and every vector cites its source.

In [ ]:
%%writefile test_crypto_vectors.py
"""Known-answer tests for the primitives used in SYSE 549 module 04.

Every vector here is published. Sources are cited per test:
  FIPS 197    Advanced Encryption Standard
  FIPS 180-4  Secure Hash Standard (SHA-1, SHA-2)
  FIPS 202    SHA-3 Standard
  SP 800-38D  GCM and GMAC
  RFC 4231    HMAC-SHA-2 test vectors
  RFC 5869    HKDF
  RFC 7748    X25519
  RFC 8032    Ed25519

Run with:      pytest -v test_crypto_vectors.py
Quiet:         pytest -q
One test:      pytest -k gcm
Stop on first: pytest -x
"""
import hashlib
import pytest

from cryptography.hazmat.primitives import hashes, hmac
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.primitives.asymmetric import x25519, ed25519
from cryptography.exceptions import InvalidTag


def h(s):
    """Hex string (whitespace allowed) to bytes -- how the standards print vectors."""
    return bytes.fromhex(s.replace(" ", "").replace("\n", ""))


# ---------------------------------------------------------------- simplest possible test
def test_fips197_aes128_appendix_c1():
    """FIPS 197 Appendix C.1: one AES-128 block, key and plaintext fixed by the standard."""
    encryptor = Cipher(algorithms.AES(h("000102030405060708090a0b0c0d0e0f")),
                       modes.ECB()).encryptor()
    ciphertext = encryptor.update(h("00112233445566778899aabbccddeeff")) + encryptor.finalize()
    assert ciphertext == h("69c4e0d86a7b0430d8cdb78070b4c55a")


# ---------------------------------------------------------------- parametrize: one test, many vectors
# Each tuple becomes its own test case with its own pass/fail line in the report.
AES_KAT = [
    ("AES-128", "000102030405060708090a0b0c0d0e0f",
     "69c4e0d86a7b0430d8cdb78070b4c55a"),
    ("AES-192", "000102030405060708090a0b0c0d0e0f1011121314151617",
     "dda97ca4864cdfe06eaf70a0ec0d7191"),
    ("AES-256", "000102030405060708090a0b0c0d0e0f101112131415161718191a1b1c1d1e1f",
     "8ea2b7ca516745bfeafc49904b496089"),
]

@pytest.mark.parametrize("name,key,expected", AES_KAT, ids=[k[0] for k in AES_KAT])
def test_fips197_all_key_sizes(name, key, expected):
    """FIPS 197 Appendix C.1-C.3: the same plaintext under all three approved key sizes."""
    encryptor = Cipher(algorithms.AES(h(key)), modes.ECB()).encryptor()
    assert (encryptor.update(h("00112233445566778899aabbccddeeff"))
            + encryptor.finalize()).hex() == expected


HASH_KAT = [
    ("sha256",   hashlib.sha256,
     "ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad"),
    ("sha512",   hashlib.sha512,
     "ddaf35a193617abacc417349ae20413112e6fa4e89a97ea20a9eeee64b55d39a"
     "2192992a274fc1a836ba3c23a3feebbd454d4423643ce80e2a9ac94fa54ca49f"),
    ("sha3_256", hashlib.sha3_256,
     "3a985da74fe225b2045c172d6bd390bd855f086e3e9d525b46bfe24511431532"),
]

@pytest.mark.parametrize("name,fn,expected", HASH_KAT, ids=[k[0] for k in HASH_KAT])
def test_hash_abc(name, fn, expected):
    """FIPS 180-4 and FIPS 202 worked examples for the message b"abc"."""
    assert fn(b"abc").hexdigest() == expected


# ---------------------------------------------------------------- fixture: build a thing once
@pytest.fixture
def gcm_vector():
    """SP 800-38D / McGrew-Viega Test Case 3. A fixture keeps the vector out of the test body."""
    return {
        "key": h("feffe9928665731c6d6a8f9467308308"),
        "iv":  h("cafebabefacedbaddecaf888"),
        "pt":  h("d9313225f88406e5a55909c5aff5269a86a7a9531534f7da2e4c303d8a318a72"
                 "1c3c0c95956809532fcf0e2449a6b525b16aedf5aa0de657ba637b391aafd255"),
        "ct":  h("42831ec2217774244b7221b784d0d49ce3aa212f2c02a4e035c17e2329aca12e"
                 "21d514b25466931c7d8f6a5aac84aa051ba30b396a0aac973d58e091473f5985"
                 "4d5c2af327cd64a62cf35abd2ba6fab4"),
    }


def test_gcm_encrypt_matches_published_vector(gcm_vector):
    """A test that asks for `gcm_vector` gets it -- pytest matches on the argument name."""
    v = gcm_vector
    assert AESGCM(v["key"]).encrypt(v["iv"], v["pt"], None) == v["ct"]


def test_gcm_decrypt_round_trip(gcm_vector):
    """The same fixture, freshly built, for a second independent test."""
    v = gcm_vector
    assert AESGCM(v["key"]).decrypt(v["iv"], v["ct"], None) == v["pt"]


# ---------------------------------------------------------------- negative test: it must FAIL
def test_gcm_rejects_tampering(gcm_vector):
    """The security property is that a modified ciphertext does NOT decrypt.

    pytest.raises asserts that the block raises -- if decrypt() returned plaintext here,
    the test fails, which is exactly what we want to know.
    """
    v = gcm_vector
    tampered = bytearray(v["ct"])
    tampered[0] ^= 0x01
    with pytest.raises(InvalidTag):
        AESGCM(v["key"]).decrypt(v["iv"], bytes(tampered), None)


def test_gcm_rejects_wrong_associated_data(gcm_vector):
    """Associated data is authenticated: changing it must break verification."""
    v = gcm_vector
    token = AESGCM(v["key"]).encrypt(v["iv"], b"value=1450", b"src=0x00;seq=0x0417")
    with pytest.raises(InvalidTag):
        AESGCM(v["key"]).decrypt(v["iv"], token, b"src=0x17;seq=0x0417")


# ---------------------------------------------------------------- MACs and KDFs
def test_rfc4231_hmac_sha256_case1():
    """RFC 4231 test case 1: 20 bytes of 0x0b as the key, b"Hi There" as the message."""
    m = hmac.HMAC(b"\x0b" * 20, hashes.SHA256())
    m.update(b"Hi There")
    assert m.finalize() == h("b0344c61d8db38535ca8afceaf0bf12b"
                             "881dc200c9833da726e9376c2e32cff7")


def test_rfc5869_hkdf_sha256_case1():
    """RFC 5869 Appendix A.1: 42 bytes of output key material."""
    okm = HKDF(algorithm=hashes.SHA256(), length=42,
               salt=h("000102030405060708090a0b0c"),
               info=h("f0f1f2f3f4f5f6f7f8f9")).derive(b"\x0b" * 22)
    assert okm == h("3cb25f25faacd57a90434f64d0362f2a2d2d0a90cf1a5a4c5db0"
                    "2d56ecc4c5bf34007208d5b887185865")


# ---------------------------------------------------------------- public key vectors
def test_rfc7748_x25519_section_6_1():
    """RFC 7748 Sec. 6.1: both public keys and the shared secret are published."""
    a = x25519.X25519PrivateKey.from_private_bytes(
        h("77076d0a7318a57d3c16c17251b26645df4c2f87ebc0992ab177fba51db92c2a"))
    b = x25519.X25519PrivateKey.from_private_bytes(
        h("5dab087e624a8a4b79e17f8b83800ee66f3bb1292618b6fd1c2f8b27ff88e0eb"))
    assert a.public_key().public_bytes_raw() == h(
        "8520f0098930a754748b7ddcb43ef75a0dbf3a0d26381af4eba4a98eaa9b4e6a")
    assert a.exchange(b.public_key()) == h(
        "4a5d9d5ba4ce2de1728e3bf480350f25e07e21c947d19e3376f09b3c1e161742")
    assert a.exchange(b.public_key()) == b.exchange(a.public_key())


def test_rfc8032_ed25519_test1_is_deterministic():
    """RFC 8032 Sec. 7.1 TEST 1. Ed25519 is deterministic, so the exact bytes are testable."""
    sk = ed25519.Ed25519PrivateKey.from_private_bytes(
        h("9d61b19deffd5a60ba844af492ec2cc44449c5697b326919703bac031cae7f60"))
    assert sk.public_key().public_bytes_raw() == h(
        "d75a980182b10ab7d54bfed3c964073a0ee172f3daa62325af021a68f707511a")
    assert sk.sign(b"") == h(
        "e5564300c360ac729086e2cc806e828a84877f1eb8e5d974d873e06522490155"
        "5fb8821590a33bacc61e39701cf9b46bd25bf5f0595bbe24655141438e7a100b")


# ---------------------------------------------------------------- xfail: a documented break
MD5_COLLISION_A = h(
    "d131dd02c5e6eec4693d9a0698aff95c2fcab58712467eab4004583eb8fb7f89"
    "55ad340609f4b30283e488832571415a085125e8f7cdc99fd91dbdf280373c5b"
    "d8823e3156348f5bae6dacd436c919c6dd53e2b487da03fd02396306d248cda0"
    "e99f33420f577ee8ce54b67080a80d1ec69821bcb6a8839396f9652b6ff72a70")
MD5_COLLISION_B = h(
    "d131dd02c5e6eec4693d9a0698aff95c2fcab50712467eab4004583eb8fb7f89"
    "55ad340609f4b30283e4888325f1415a085125e8f7cdc99fd91dbd7280373c5b"
    "d8823e3156348f5bae6dacd436c919c6dd53e23487da03fd02396306d248cda0"
    "e99f33420f577ee8ce54b67080280d1ec69821bcb6a8839396f965ab6ff72a70")


@pytest.mark.xfail(strict=True,
                   reason="MD5 collision resistance is broken (Wang et al., 2004)")
def test_md5_is_collision_resistant():
    """Asserts the property MD5 is SUPPOSED to have. It does not have it.

    strict=True means the suite FAILS if this ever unexpectedly passes -- which would
    mean the vectors were edited. This is how you keep a known break documented and
    under test instead of deleting the evidence.
    """
    assert hashlib.md5(MD5_COLLISION_A).digest() != hashlib.md5(MD5_COLLISION_B).digest()


def test_sha256_separates_the_md5_collision_pair():
    """The same two inputs, under a hash that is not broken. This one must pass."""
    assert MD5_COLLISION_A != MD5_COLLISION_B
    assert hashlib.sha256(MD5_COLLISION_A).digest() != hashlib.sha256(MD5_COLLISION_B).digest()


### Running it

From a terminal, in this folder, the whole command is `pytest`. From inside the notebook we
call the same thing through `sys.executable` so it uses this kernel's interpreter on any
operating system.

Read the summary line first, then the letters: `.` passed, `F` failed, `x` xfailed
(expected failure), `X` xpassed (unexpectedly passed -- suspicious), `s` skipped,
`E` error during setup.

In [ ]:
import subprocess, sys

result = subprocess.run([sys.executable, "-m", "pytest", "-v", "test_crypto_vectors.py"],
                        capture_output=True, text=True)
print(result.stdout[-4000:])
print("exit code:", result.returncode, "  (0 = all good; anything else fails a CI build)")

Notice what `parametrize` bought us: `test_fips197_all_key_sizes[AES-192]` is its own
line in the report. When a vector fails you are told *which* one, without reading a
traceback -- and that is the difference between "the crypto is broken somewhere" and
"AES-192 is broken."

Notice also the `xfail` line for MD5. The suite is green, and the broken property is still
documented and still executed.

### What a failure actually looks like

This is the feature that makes pytest worth using. Write a test with a wrong expected
value and pytest shows you both sides of the comparison, with the differing region marked.

We write this one to a **temporary directory** on purpose: a deliberately failing test left
in the course folder would be collected by `pytest` every time you ran the real suite.

In [ ]:
import tempfile, pathlib, subprocess, sys

demo_dir = pathlib.Path(tempfile.mkdtemp(prefix='pytest_demo_'))

# Build the deliberately-wrong test as a list of lines, so nothing here is executed
# until pytest runs it in that temporary directory.
broken = chr(10).join([
    'import hashlib',
    '',
    'def test_sha256_with_a_typo_in_the_expected_value():',
    '    # The final character of the expected digest was changed from d to e.',
    '    assert hashlib.sha256(b\'abc\').hexdigest() == (',
    '        \'ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ae\')',
    '',
    'def test_digest_length():',
    '    # SHA-256 is 32 bytes, not 16. Another wrong expectation, on purpose.',
    '    assert len(hashlib.sha256(b\'abc\').digest()) == 16',
])
(demo_dir / 'test_broken.py').write_text(broken)

result = subprocess.run([sys.executable, '-m', 'pytest', '-q', str(demo_dir)],
                        capture_output=True, text=True)
print(result.stdout[-3000:])

pytest rewrote the `assert` so it could print the actual value, the expected value, and a
`- ` / `+ ` diff pointing at the character that differs. No logging, no `print`, no
`assertEqual` with a hand-written message.

### Scaling up: vectors from a file

Hard-coding vectors in the test file works for a dozen. NIST's CAVP response files and the
Wycheproof JSON sets contain thousands, so the real pattern is to *read* the vectors and
parametrize over what you read. The structure is the same either way.

In [ ]:
%%writefile test_vectors_from_file.py
"""Parametrizing over an external vector file -- the pattern used for CAVP and Wycheproof.

The loader below reads a small JSON file. Point it at a real CAVP .rsp or a Wycheproof
test group and only the parsing changes; the test body does not.
"""
import json
import pathlib

import pytest
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes

VECTOR_FILE = pathlib.Path(__file__).with_name("aes_ecb_vectors.json")


def load_vectors():
    """Read the vector file at collection time so each case becomes its own test."""
    if not VECTOR_FILE.exists():
        return []
    return json.loads(VECTOR_FILE.read_text())["tests"]


VECTORS = load_vectors()


@pytest.mark.skipif(not VECTORS, reason="aes_ecb_vectors.json not found")
@pytest.mark.parametrize("case", VECTORS, ids=lambda c: c["id"])
def test_aes_ecb_vector(case):
    """One test per entry in the file. Add a row to the JSON, get a new test."""
    encryptor = Cipher(algorithms.AES(bytes.fromhex(case["key"])), modes.ECB()).encryptor()
    got = encryptor.update(bytes.fromhex(case["plaintext"])) + encryptor.finalize()
    assert got.hex() == case["ciphertext"]


In [ ]:
# Create the vector file the test above reads. In practice this would be a CAVP download.
import json, pathlib

vectors = {
    "source": "FIPS 197 Appendix C (AES-128/192/256 single-block known-answer tests)",
    "tests": [
        {"id": "FIPS197-C1-AES128",
         "key": "000102030405060708090a0b0c0d0e0f",
         "plaintext": "00112233445566778899aabbccddeeff",
         "ciphertext": "69c4e0d86a7b0430d8cdb78070b4c55a"},
        {"id": "FIPS197-C2-AES192",
         "key": "000102030405060708090a0b0c0d0e0f1011121314151617",
         "plaintext": "00112233445566778899aabbccddeeff",
         "ciphertext": "dda97ca4864cdfe06eaf70a0ec0d7191"},
        {"id": "FIPS197-C3-AES256",
         "key": "000102030405060708090a0b0c0d0e0f101112131415161718191a1b1c1d1e1f",
         "plaintext": "00112233445566778899aabbccddeeff",
         "ciphertext": "8ea2b7ca516745bfeafc49904b496089"},
    ],
}
pathlib.Path("aes_ecb_vectors.json").write_text(json.dumps(vectors, indent=2))
print("wrote aes_ecb_vectors.json with", len(vectors["tests"]), "vectors")

In [ ]:
# Run just the file-driven suite. The ids come straight from the vector file.
result = subprocess.run([sys.executable, "-m", "pytest", "-v", "test_vectors_from_file.py"],
                        capture_output=True, text=True)
print(result.stdout[-2500:])

### The commands worth memorizing

| Command | What it does |
|---|---|
| `pytest` | Discover and run everything under the current directory |
| `pytest -v` | One line per test, with the parametrize id |
| `pytest -q` | Quiet: just the dots and the summary |
| `pytest -x` | Stop at the first failure |
| `pytest -k gcm` | Only tests whose name matches `gcm` |
| `pytest --tb=short` | Shorter tracebacks |
| `pytest --lf` | Re-run only the tests that failed last time |
| `pytest --collect-only` | List what *would* run, without running it |

Two files you will meet: `conftest.py` holds fixtures shared across test files, and
`pytest.ini` / `pyproject.toml` holds project-wide settings. Neither is needed to start.

### Why the exit code matters

`pytest` returns 0 when everything passed and non-zero otherwise. That is the entire
contract a build system needs. A GitHub Actions job is about six lines:

```yaml
- run: pip install cryptography pytest
- run: pytest -q
```

From that point on, nobody can merge a change that breaks a published test vector without
being told. That is the mechanism that turns "we validated the cryptography" from a claim
made once, in a meeting, into a property that is checked on every change.

### Exercises

1. Add the AES-CBC vectors from **SP 800-38A Appendix F.2.1** to `test_crypto_vectors.py`
   as a new parametrized test. Run `pytest -k cbc` to run only those.
2. Break one hex digit of an expected value on purpose. Read the failure output and say
   exactly which byte position differs.
3. `test_md5_is_collision_resistant` is marked `xfail(strict=True)`. Change the two
   collision blocks so they are no longer a collision, re-run, and explain the `XPASS`
   failure you get and why `strict=True` is the right setting for a security test.
4. Download one CAVP `.rsp` file for AES-ECB, write a parser for it, and point
   `test_vectors_from_file.py` at the result. How many tests does `pytest --collect-only`
   report?
5. Write a test that would fail if someone replaced `os.urandom` with a fixed value.
   (Hint: you cannot test randomness directly -- test the property you actually need.)

---
## Where the official vectors come from

| Source | What it holds |
|---|---|
| **CAVP** -- <https://csrc.nist.gov/projects/cryptographic-algorithm-validation-program> | Response files (`.rsp`) for AES, SHA, HMAC, CMAC, ECDSA, DRBG and more |
| **ACVP** -- <https://github.com/usnistgov/ACVP> | The current JSON-based automated test framework, including PQC |
| **FIPS documents** -- <https://csrc.nist.gov/publications/fips> | 197 (AES), 180-4 (SHA-2), 202 (SHA-3), 186-5 (signatures), 198-1 (HMAC), 203/204/205 (PQC) |
| **NIST SP 800 series** -- <https://csrc.nist.gov/publications/sp800> | 38A-38F (modes), 56A/B/C (key establishment), 57 (key management), 90A/B/C (RNG), 131A (transitions), 132 (PBKDF) |
| **IETF RFCs** -- <https://www.rfc-editor.org/> | 4231 (HMAC), 5869 (HKDF), 6070/7914 (PBKDF2), 6979 (deterministic ECDSA), 7748 (X25519), 8017 (PKCS#1), 8032 (EdDSA), 8439 (ChaCha20-Poly1305), 8452 (GCM-SIV), 9180 (HPKE) |
| **Project Wycheproof** -- <https://github.com/C2SP/wycheproof> | Vectors designed to catch *implementation* bugs (edge cases, malleable signatures, invalid curve points) that the standards' happy-path vectors miss |

## Reading a validation certificate

A CAVP certificate says a specific implementation, on a specific platform, with a specific
build, reproduced the required vectors on a given date. That is a real and useful fact. It
is also narrower than most people assume:

* It covers the **algorithm**, not the **protocol**. A validated AES core says nothing
  about whether the surrounding code reuses a GCM nonce.
* It covers the **implementation as tested**. A different compiler, a different
  optimization level, or a different chip may not be covered.
* It says nothing about **key management** -- where keys are stored, how they are
  provisioned, or whether the same key ships on every unit.
* It says nothing about **side channels** unless the module was also tested for them.

So the useful supplier questions are: *what is the certificate number, what operational
environment does it name, and which specific integration mistakes -- nonce reuse, a key
shipped identically on every unit, acting on unauthenticated plaintext -- does your design
prevent?*

## Exercises

1. Download one CAVP `.rsp` file for AES-CBC and write a loop that runs every vector in it
   through `cryptography`. How many vectors are in the file?
2. Add a Wycheproof ECDSA test group to this notebook. Which cases fail on a naive verifier
   that does not check signature malleability?
3. A supplier gives you a CAVP certificate for their AES-GCM core. Write the three follow-up
   questions you would ask before accepting it as evidence that their product is secure.
4. Take one algorithm this notebook does not cover -- AES-CCM, or SLH-DSA -- find its
   published vectors, and add a `check()` for it.